# DSA 4060 – Personalized Movie Recommender

**Name:** Abdiqalaq Issack  
**Student ID:** 671243  
**Assigned User ID:** 4

## Practical Quiz

This notebook develops a simple personalized movie recommender for User 4 using historical movie ratings and movie genre information.

In [51]:
import pandas as pd
import numpy as np

# Load datasets
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")

print("Movies dataset loaded successfully.")
print("Ratings dataset loaded successfully.")

Movies dataset loaded successfully.
Ratings dataset loaded successfully.


## Task 1 – Load and Inspect the Data

This section examines the structure, dimensions, and data quality of the movies and ratings datasets.

In [52]:
# Display the first five rows of each dataset

print("First 5 rows of movies dataset:")
display(movies.head())

print("\nFirst 5 rows of ratings dataset:")
display(ratings.head())

First 5 rows of movies dataset:


,movie_id,title,genres
0,1,Inception,Sci-Fi|Thriller
1,2,Interstellar,Sci-Fi|Drama
2,3,The Matrix,Sci-Fi|Action
3,4,Arrival,Sci-Fi|Drama
4,5,Edge of Tomorrow,Sci-Fi|Action



First 5 rows of ratings dataset:


,user_id,movie_id,rating
0,1,2,3.5
1,1,3,4.5
2,1,5,3.5
3,1,6,3.5
4,1,11,3.0


### 1.1 Previewing the Datasets

- The first five rows of each dataset are displayed to understand their structure and the variables available for building the recommender system.
- The movies dataset contains information about each movie, while the ratings dataset records the ratings given by users to different movies.

In [53]:
# Check the number of rows and columns

print("Movies dataset shape:", movies.shape)
print("Number of movie rows:", movies.shape[0])
print("Number of movie columns:", movies.shape[1])

print("\nRatings dataset shape:", ratings.shape)
print("Number of rating rows:", ratings.shape[0])
print("Number of rating columns:", ratings.shape[1])

Movies dataset shape: (36, 3)
Number of movie rows: 36
Number of movie columns: 3

Ratings dataset shape: (440, 3)
Number of rating rows: 440
Number of rating columns: 3


### 1.2 Dataset Dimensions

- The dimensions of each dataset are examined to determine the number of observations (rows) and variables (columns) available for the analysis.

In [54]:
# Check for missing values

print("Missing values in movies dataset:")
display(movies.isnull().sum())

print("\nMissing values in ratings dataset:")
display(ratings.isnull().sum())

Missing values in movies dataset:


movie_id    0
title       0
genres      0
dtype: int64


Missing values in ratings dataset:


user_id     0
movie_id    0
rating      0
dtype: int64

### 1.3 Missing Values

- Missing values are checked in both datasets because incomplete movie information or ratings could affect the user profiling and recommendation process.

## Task 2 – Profile the Assigned User

The assigned user for Student ID **671243** is **User 4**. In this section, the user's historical ratings are examined to identify highly rated movies and determine their apparent genre preferences.

### 2.1 Movies Rated by User 4

The ratings dataset is filtered to include only ratings made by User 4. These ratings are then merged with the movies dataset so that each rating can be associated with its movie title and genres.

In [55]:
# Filter ratings for the assigned user
assigned_user_id = 4

user_ratings = ratings[ratings["user_id"] == assigned_user_id].copy()

# Merge with movie information
user_profile = user_ratings.merge(
    movies,
    on="movie_id",
    how="left"
)

# Display the movies rated by User 4
user_profile = user_profile[
    ["movie_id", "title", "genres", "rating"]
].sort_values(by="rating", ascending=False)

print(f"Number of movies rated by User {assigned_user_id}: {len(user_profile)}")
display(user_profile)

Number of movies rated by User 4: 13


,movie_id,title,genres,rating
7,26,A Quiet Place,Horror|Thriller,4.5
6,25,Get Out,Horror|Thriller,3.5
0,1,Inception,Sci-Fi|Thriller,3.0
1,8,John Wick,Action|Thriller,3.0
2,14,The Notebook,Romance|Drama,3.0
3,17,The Pursuit of Happyness,Drama|Biography,3.0
5,21,Toy Story,Animation|Comedy,3.0
12,34,Creed,Sports|Drama,3.0
4,20,Hidden Figures,Drama|Biography,2.5
8,28,Knives Out,Mystery|Comedy,2.5


### 2.2 Three Highest-Rated Movies

The movies are ranked by rating to identify three movies that provide the strongest evidence of User 4's preferences. In the case of tied ratings, any three movies among the highest-rated movies are valid.

In [56]:
# Identify the three highest-rated movies
top_3_movies = user_profile.nlargest(3, "rating")

print("User 4's three highest-rated movies:")
display(top_3_movies[["title", "genres", "rating"]])

User 4's three highest-rated movies:


,title,genres,rating
7,A Quiet Place,Horror|Thriller,4.5
6,Get Out,Horror|Thriller,3.5
0,Inception,Sci-Fi|Thriller,3.0


### 2.3 Genre Preference Analysis

To identify User 4's apparent genre preferences, the genre labels of rated movies are separated and analysed together with the user's ratings. The average rating and number of rated movies for each genre provide evidence of which genres the user appears to prefer.

In [57]:
# Separate the genres for each rated movie
genre_analysis = user_profile.copy()

genre_analysis["genres"] = genre_analysis["genres"].str.split("|")
genre_analysis = genre_analysis.explode("genres")

# Calculate rating statistics for each genre
genre_preferences = (
    genre_analysis
    .groupby("genres")
    .agg(
        movies_rated=("rating", "count"),
        average_rating=("rating", "mean")
    )
    .sort_values(
        by=["average_rating", "movies_rated"],
        ascending=[False, False]
    )
)

genre_preferences["average_rating"] = genre_preferences["average_rating"].round(2)

display(genre_preferences)

,movies_rated,average_rating
genres,,
Horror,2,4.00
Thriller,4,3.50
Animation,1,3.00
Romance,1,3.00
Sci-Fi,1,3.00
Sports,1,3.00
Drama,4,2.88
Action,2,2.75
Biography,2,2.75


### Interpretation of User Preferences

- User 4 shows the strongest preference for **Horror**, which has the highest average rating of **4.00** across two rated movies. **Thriller** also appears to be an important preference because it occurs in four rated movies and has a relatively high average rating of **3.50**. 
- The user's highest-rated movie, *A Quiet Place* (4.5), combines both Horror and Thriller, providing strong evidence that movies containing these genres are particularly relevant to this user.

## Task 3 – Build the Recommendation Logic

A content-based recommendation approach is used to recommend movies based on genre similarity. Movies rated **4.0 or higher** are treated as movies that User 4 likes. The genres are converted into numerical features, and cosine similarity is used to measure how similar the remaining movies are to the user's positively rated movies.

### 3.1 Identify Positively Rated Movies

According to the quiz requirements, a movie is considered positively rated when the assigned user gives it a rating of **4.0 or higher**. These movies form the preference profile used by the recommender.

In [58]:
# Movies liked by User 4 (rating >= 4.0)
liked_movies = user_profile[user_profile["rating"] >= 4.0].copy()

print(f"Number of positively rated movies: {len(liked_movies)}")
display(liked_movies[["movie_id", "title", "genres", "rating"]])

Number of positively rated movies: 1


,movie_id,title,genres,rating
7,26,A Quiet Place,Horror|Thriller,4.5


In [59]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### 3.2 Convert Genres into Numerical Features

- `CountVectorizer` is used to transform the movie genre labels into numerical feature vectors. 
- Each genre becomes a feature indicating whether it is associated with a movie. 
- This representation allows genre similarity between movies to be calculated mathematically.

In [60]:
# Prepare genre text for vectorization
movies_features = movies.copy()

# Replace "|" with spaces so CountVectorizer treats genres as separate terms
movies_features["genre_text"] = movies_features["genres"].str.replace("|", " ", regex=False)

# Convert genre information into numerical vectors
vectorizer = CountVectorizer()
genre_matrix = vectorizer.fit_transform(movies_features["genre_text"])

print("Genre feature matrix shape:", genre_matrix.shape)
print("Genre features:")
print(vectorizer.get_feature_names_out())

Genre feature matrix shape: (36, 17)
Genre features:
['action' 'adventure' 'animation' 'biography' 'comedy' 'crime' 'drama'
 'family' 'fi' 'horror' 'musical' 'mystery' 'romance' 'sci' 'sports'
 'superhero' 'thriller']


### 3.3 Create the User Preference Profile

- The genre vectors of movies positively rated by User 4 are used to construct the user's preference profile. 
- Because the user's positive ratings represent their strongest observed preferences, this profile provides the reference against which candidate movies are compared.

### 3.4 Calculate Cosine Similarity

- Cosine similarity measures how closely each movie's genre vector matches User 4's preference profile. 
- A higher similarity score indicates a stronger genre-based match with the movies the user positively rated.

In [61]:
# Locate positively rated movies in the movie catalogue
liked_indices = movies_features.index[
    movies_features["movie_id"].isin(liked_movies["movie_id"])
].tolist()

# Build the user's genre preference vector
user_preference_vector = genre_matrix[liked_indices].mean(axis=0)

# Convert numpy matrix to a standard NumPy array
user_preference_vector = np.asarray(user_preference_vector)

print("User preference vector created from:")
display(liked_movies[["title", "genres", "rating"]])

User preference vector created from:


,title,genres,rating
7,A Quiet Place,Horror|Thriller,4.5


In [62]:
# Calculate similarity between every movie and the user preference profile
similarity_scores = cosine_similarity(
    genre_matrix,
    user_preference_vector
).flatten()

movies_features["similarity_score"] = similarity_scores

# IDs of movies already rated by User 4
rated_movie_ids = set(user_profile["movie_id"])

# Remove movies already rated by the user
candidate_movies = movies_features[
    ~movies_features["movie_id"].isin(rated_movie_ids)
].copy()

# Rank unseen movies by similarity
recommendations = (
    candidate_movies
    .sort_values(
        by=["similarity_score", "title"],
        ascending=[False, True]
    )
    .head(5)
)

print("Top 5 recommendations for User 4:")
display(
    recommendations[
        ["movie_id", "title", "genres", "similarity_score"]
    ]
)

Top 5 recommendations for User 4:


,movie_id,title,genres,similarity_score
26,27,The Conjuring,Horror,0.707107
11,12,Mission: Impossible - Fallout,Action|Thriller,0.500000
3,4,Arrival,Sci-Fi|Drama,0.000000
10,11,Avengers: Endgame,Action|Superhero,0.000000
9,10,Black Panther,Action|Superhero,0.000000


### Recommendation Logic Interpretation

- The strongest recommendation is *The Conjuring*, with a cosine similarity score of approximately **0.707**, because it shares the Horror genre with User 4's positively rated movie, *A Quiet Place*. *Mission: Impossible - Fallout* has a similarity score of **0.500** because it shares the Thriller genre with *A Quiet Place*.
-  The remaining recommendations have similarity scores of 0 because they do not directly share the Horror or Thriller features represented in the current positive-preference profile; however, they are the next eligible unseen movies after ranking the candidate set.

## Task 4 – Explain the Recommendations

The following table explains why each of the five unseen movies was returned by the content-based recommender. The explanations are based on genre overlap with User 4's positive preference profile and the resulting cosine similarity scores.

In [63]:
# Create explanations for the recommendations
def recommendation_reason(row):
    if row["similarity_score"] > 0:
        shared_genres = set(row["genres"].split("|")) & {"Horror", "Thriller"}
        
        return (
            f"Shares {', '.join(sorted(shared_genres))} genre preference(s) "
            f"with the user's positively rated movie A Quiet Place; "
            f"cosine similarity = {row['similarity_score']:.3f}."
        )
    else:
        return (
            "No direct Horror/Thriller genre overlap with the current "
            "positive-preference profile; included among the highest-ranked "
            "remaining unseen movies after similarity ranking."
        )

results_table = recommendations.copy()

results_table["Reason"] = results_table.apply(
    recommendation_reason,
    axis=1
)

results_table.insert(
    0,
    "Rank",
    range(1, len(results_table) + 1)
)

results_table = results_table[
    ["Rank", "title", "genres", "Reason"]
].rename(
    columns={
        "title": "Recommended Movie",
        "genres": "Genres"
    }
)

display(results_table)

,Rank,Recommended Movie,Genres,Reason
26,1,The Conjuring,Horror,Shares Horror genre preference(s) with the use...
11,2,Mission: Impossible - Fallout,Action|Thriller,Shares Thriller genre preference(s) with the u...
3,3,Arrival,Sci-Fi|Drama,No direct Horror/Thriller genre overlap with t...
10,4,Avengers: Endgame,Action|Superhero,No direct Horror/Thriller genre overlap with t...
9,5,Black Panther,Action|Superhero,No direct Horror/Thriller genre overlap with t...


### Recommendation Logic Interpretation

- The strongest recommendation is *The Conjuring*, with a cosine similarity score of approximately **0.707**, because it shares the Horror genre with User 4's positively rated movie, *A Quiet Place*. *Mission: 
- Impossible - Fallout* has a similarity score of **0.500** because it shares the Thriller genre with *A Quiet Place*. 
- The remaining recommendations have similarity scores of 0 because they do not directly share the Horror or Thriller features represented in the current positive-preference profile; however, they are the next eligible unseen movies after ranking the candidate set.

## Task 5 – Critical Thinking

### Limitation

- A limitation of this recommender is that it relies only on movie genres and on movies that User 4 rated at least 4.0. 
- In this case, only *A Quiet Place* qualifies as a positively rated movie, so the user's preference profile is based mainly on Horror and Thriller. 
- This produces a narrow profile and causes several candidate movies to receive a cosine similarity score of zero.

### Suggested Improvement

- A realistic improvement would be to build a hybrid recommender that combines content-based genre similarity with collaborative filtering. Ratings from users with similar preferences could provide additional evidence about movies User 4 may enjoy, allowing the system to generate more personalized recommendations even when the user has only a small number of highly rated movies.